In [1]:
#setup + imports

import sys
sys.path.append('..')

import pandas as pd
import os
import re
from config import engine

# Helper function to extract uid from filename
def extract_uid(filename):
    """Extract uid from filenames like 'activity_u01.csv' -> 'u01'"""
    match = re.search(r'u\d+', filename)
    return match.group() if match else None

print("✓ Setup complete")

✓ Setup complete


In [3]:
# Check EMA table schemas
print("EMA_DEFINITION columns:")
result = pd.read_sql("SELECT * FROM ema_definition LIMIT 0", engine)
print(result.columns.tolist())

print("\nEMA_RESPONSE columns:")
result = pd.read_sql("SELECT * FROM ema_response LIMIT 0", engine)
print(result.columns.tolist())

EMA_DEFINITION columns:
['question_id', 'question_name', 'question_text', 'options']

EMA_RESPONSE columns:
['response_id', 'uid', 'question_id', 'timestamp', 'latitude', 'longitude', 'response_value']


In [4]:
# Explore EMA data structure
ema_dir = '/Users/test/Downloads/Data Management & Databases/studentlife_project/data/EMA'

# Check what's in the EMA directory
print("EMA directory contents:")
items = os.listdir(ema_dir)
for item in sorted(items)[:10]:  # Show first 10
    print(f"  {item}")

print(f"\nTotal items: {len(items)}")

# Look at one example file to understand structure
example_files = [f for f in items if f.endswith('.json')]
if example_files:
    print(f"\nExample file: {example_files[0]}")
    
    import json
    filepath = os.path.join(ema_dir, example_files[0])
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    print("\nFile structure (first entry):")
    if isinstance(data, list) and len(data) > 0:
        print(json.dumps(data[0], indent=2))
    elif isinstance(data, dict):
        print(json.dumps(data, indent=2)[:500])  # First 500 chars

EMA directory contents:
  .DS_Store
  EMA_definition.json
  response

Total items: 3

Example file: EMA_definition.json

File structure (first entry):
{
  "name": "Social",
  "questions": [
    {
      "options": "[1]0-4 persons, [2]5-9 persons, [3]10-19 persons, [4]20-49 persons, [5]50-99 persons, [6]over 100 persons, ",
      "question_id": "number",
      "question_text": "How many people did you have contact with yesterday, including anyone you said hello to, chatted, talked or discussed matters with, whether you did it face-to-face, by telephone, by mail or on the internet, and whether you personally knew the person or not? Please select one of the following categories that best matches your estimate:"
    },
    {
      "options": "",
      "question_id": "location",
      "question_text": ""
    }
  ]
}


In [5]:
# Check response directory structure
response_dir = os.path.join(ema_dir, 'response')

print("Response directory contents:")
response_files = os.listdir(response_dir)
print(f"Total files: {len(response_files)}")
print("\nFirst 10 files:")
for f in sorted(response_files)[:10]:
    print(f"  {f}")

# Look at one response file
if response_files:
    example_response = [f for f in response_files if f.endswith('.json')][0]
    print(f"\nExample response file: {example_response}")
    
    import json
    filepath = os.path.join(response_dir, example_response)
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    print("\nResponse file structure (first 2 entries):")
    if isinstance(data, list):
        for i, entry in enumerate(data[:2]):
            print(f"\nEntry {i+1}:")
            print(json.dumps(entry, indent=2))

Response directory contents:
Total files: 27

First 10 files:
  .DS_Store
  Activity
  Administration response
  Behavior
  Boston Bombing
  Cancelled Classes
  Class
  Class 2
  Comment
  Dartmouth now


IndexError: list index out of range

In [6]:
# Check what's inside the response subdirectories
response_dir = os.path.join(ema_dir, 'response')

# Pick a folder to explore
example_folder = 'Activity'
activity_path = os.path.join(response_dir, example_folder)

print(f"Contents of '{example_folder}' folder:")
if os.path.isdir(activity_path):
    files = os.listdir(activity_path)
    print(f"Total files: {len(files)}")
    print("\nFirst 10 files:")
    for f in sorted(files)[:10]:
        print(f"  {f}")
    
    # Try to read one file
    if files:
        # Find a file that's not .DS_Store
        actual_files = [f for f in files if not f.startswith('.')]
        if actual_files:
            example_file = actual_files[0]
            print(f"\nExample file: {example_file}")
            
            filepath = os.path.join(activity_path, example_file)
            
            # Try reading as JSON
            try:
                import json
                with open(filepath, 'r') as f:
                    data = json.load(f)
                print("\nFile contents (first 2 entries):")
                if isinstance(data, list):
                    for i, entry in enumerate(data[:2]):
                        print(f"\nEntry {i+1}:")
                        print(json.dumps(entry, indent=2))
            except:
                # Try reading as text
                with open(filepath, 'r') as f:
                    content = f.read()[:500]
                print("\nFile contents (first 500 chars):")
                print(content)

Contents of 'Activity' folder:
Total files: 49

First 10 files:
  Activity_u00.json
  Activity_u01.json
  Activity_u02.json
  Activity_u03.json
  Activity_u04.json
  Activity_u05.json
  Activity_u07.json
  Activity_u08.json
  Activity_u09.json
  Activity_u10.json

Example file: Activity_u56.json

File contents (first 2 entries):

Entry 1:
{
  "Social2": "1",
  "null": "1",
  "resp_time": 1365393298
}

Entry 2:
{
  "other_relaxing": "4",
  "other_working": "1",
  "relaxing": "1",
  "resp_time": 1365396239,
  "working": "1"
}


In [7]:
# ===========================
# EMA DATA LOADING
# ===========================

import json

ema_dir = '/Users/test/Downloads/Data Management & Databases/studentlife_project/data/EMA'

# ----------------------------
# 1. EMA_DEFINITION
# ----------------------------
def load_ema_definition():
    """Load EMA question definitions"""
    filepath = os.path.join(ema_dir, 'EMA_definition.json')
    
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    definitions = []
    
    # Each entry has a name and list of questions
    for category in data:
        category_name = category.get('name', 'Unknown')
        questions = category.get('questions', [])
        
        for q in questions:
            question_id = q.get('question_id')
            question_text = q.get('question_text', '')
            options = q.get('options', '')
            
            # Only add if question_id exists and is not empty
            if question_id:
                definitions.append({
                    'question_id': question_id,
                    'question_name': category_name,
                    'question_text': question_text,
                    'options': options
                })
    
    df = pd.DataFrame(definitions)
    df.to_sql('ema_definition', engine, if_exists='append', index=False)
    
    print(f"✓ Loaded {len(df):,} EMA question definitions")
    return len(df)

# ----------------------------
# 2. EMA_RESPONSE
# ----------------------------
def load_ema_responses():
    """Load all EMA responses from all categories"""
    response_dir = os.path.join(ema_dir, 'response')
    
    all_responses = []
    
    # Get all response category folders
    categories = [d for d in os.listdir(response_dir) if not d.startswith('.')]
    
    print(f"Found {len(categories)} EMA response categories")
    
    for category in categories:
        category_path = os.path.join(response_dir, category)
        
        if not os.path.isdir(category_path):
            continue
        
        print(f"  Processing {category}...")
        
        # Get all JSON files in this category
        files = [f for f in os.listdir(category_path) if f.endswith('.json')]
        
        for filename in files:
            uid = extract_uid(filename)
            if not uid:
                continue
            
            filepath = os.path.join(category_path, filename)
            
            try:
                with open(filepath, 'r') as f:
                    data = json.load(f)
                
                # Each file contains a list of response entries
                for entry in data:
                    resp_time = entry.get('resp_time')
                    latitude = entry.get('latitude')
                    longitude = entry.get('longitude')
                    
                    # All other keys are question_id: response_value pairs
                    for key, value in entry.items():
                        if key not in ['resp_time', 'latitude', 'longitude']:
                            all_responses.append({
                                'uid': uid,
                                'question_id': key,
                                'timestamp': resp_time,
                                'latitude': latitude,
                                'longitude': longitude,
                                'response_value': str(value) if value is not None else None
                            })
            
            except Exception as e:
                print(f"    ⚠ Error loading {filename}: {e}")
                continue
    
    df = pd.DataFrame(all_responses)
    df.to_sql('ema_response', engine, if_exists='append', index=False)
    
    print(f"\n✓ Loaded {len(df):,} EMA responses")
    return len(df)

# ----------------------------
# RUN EMA LOADING
# ----------------------------
print("="*60)
print("LOADING EMA DATA")
print("="*60 + "\n")

try:
    load_ema_definition()
except Exception as e:
    print(f"❌ Error loading ema_definition: {e}\n")

try:
    load_ema_responses()
except Exception as e:
    print(f"❌ Error loading ema_responses: {e}\n")

print("\n" + "="*60)
print("EMA DATA LOADING COMPLETE!")
print("="*60)

LOADING EMA DATA

❌ Error loading ema_definition: Execution failed on sql 'INSERT INTO ema_definition (question_id, question_name, question_text, options) VALUES (:question_id, :question_name, :question_text, :options)': (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "ema_definition_pkey"
DETAIL:  Key (question_id)=(location) already exists.

[SQL: INSERT INTO ema_definition (question_id, question_name, question_text, options) VALUES (%(question_id__0)s, %(question_name__0)s, %(question_text__0)s, %(options__0)s), (%(question_id__1)s, %(question_name__1)s, %(question_text__1)s, %(options__1)s), ... 8565 characters truncated ... ns__101)s), (%(question_id__102)s, %(question_name__102)s, %(question_text__102)s, %(options__102)s)]
[parameters: {'question_text__0': 'How many people did you have contact with yesterday, including anyone you said hello to, chatted, talked or discussed matters with, whether you did i ... (38 characters truncated) ... or on the

In [8]:
# ===========================
# EMA DATA LOADING (FIXED)
# ===========================

import json

ema_dir = '/Users/test/Downloads/Data Management & Databases/studentlife_project/data/EMA'

# ----------------------------
# 1. EMA_DEFINITION (FIXED - Handle duplicates)
# ----------------------------
def load_ema_definition():
    """Load EMA question definitions"""
    filepath = os.path.join(ema_dir, 'EMA_definition.json')
    
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    definitions = {}  # Use dict to avoid duplicates
    
    for category in data:
        category_name = category.get('name', 'Unknown')
        questions = category.get('questions', [])
        
        for q in questions:
            question_id = q.get('question_id')
            question_text = q.get('question_text', '')
            options = q.get('options', '')
            
            # Only add if question_id exists and not already in dict
            if question_id and question_id not in definitions:
                definitions[question_id] = {
                    'question_id': question_id,
                    'question_name': category_name,
                    'question_text': question_text,
                    'options': options
                }
    
    df = pd.DataFrame(list(definitions.values()))
    df.to_sql('ema_definition', engine, if_exists='append', index=False)
    
    print(f"✓ Loaded {len(df):,} unique EMA question definitions")
    return len(df)

# ----------------------------
# 2. EMA_RESPONSE (FIXED - Skip null and invalid question_ids)
# ----------------------------
def load_ema_responses():
    """Load all EMA responses from all categories"""
    response_dir = os.path.join(ema_dir, 'response')
    
    # First, get valid question_ids from database
    valid_questions = pd.read_sql("SELECT question_id FROM ema_definition", engine)
    valid_question_ids = set(valid_questions['question_id'].tolist())
    
    print(f"Found {len(valid_question_ids)} valid question IDs in ema_definition")
    
    all_responses = []
    skipped_count = 0
    
    categories = [d for d in os.listdir(response_dir) if not d.startswith('.')]
    
    print(f"Found {len(categories)} EMA response categories\n")
    
    for category in categories:
        category_path = os.path.join(response_dir, category)
        
        if not os.path.isdir(category_path):
            continue
        
        print(f"  Processing {category}...")
        
        files = [f for f in os.listdir(category_path) if f.endswith('.json')]
        
        for filename in files:
            uid = extract_uid(filename)
            if not uid:
                continue
            
            filepath = os.path.join(category_path, filename)
            
            try:
                with open(filepath, 'r') as f:
                    data = json.load(f)
                
                for entry in data:
                    resp_time = entry.get('resp_time')
                    latitude = entry.get('latitude')
                    longitude = entry.get('longitude')
                    
                    for key, value in entry.items():
                        if key not in ['resp_time', 'latitude', 'longitude']:
                            # Skip if question_id is null or not in valid set
                            if key == 'null' or key not in valid_question_ids:
                                skipped_count += 1
                                continue
                            
                            all_responses.append({
                                'uid': uid,
                                'question_id': key,
                                'timestamp': resp_time,
                                'latitude': latitude,
                                'longitude': longitude,
                                'response_value': str(value) if value is not None else None
                            })
            
            except Exception as e:
                print(f"    ⚠ Error loading {filename}: {e}")
                continue
    
    df = pd.DataFrame(all_responses)
    df.to_sql('ema_response', engine, if_exists='append', index=False)
    
    print(f"\n✓ Loaded {len(df):,} EMA responses")
    print(f"  (Skipped {skipped_count:,} responses with invalid/null question_ids)")
    return len(df)

# ----------------------------
# RUN EMA LOADING
# ----------------------------
print("="*60)
print("LOADING EMA DATA (FIXED)")
print("="*60 + "\n")

try:
    load_ema_definition()
except Exception as e:
    print(f"❌ Error loading ema_definition: {e}\n")

try:
    load_ema_responses()
except Exception as e:
    print(f"❌ Error loading ema_responses: {e}\n")

print("\n" + "="*60)
print("EMA DATA LOADING COMPLETE!")
print("="*60)

LOADING EMA DATA (FIXED)

✓ Loaded 65 unique EMA question definitions
Found 65 valid question IDs in ema_definition
Found 26 EMA response categories

  Processing Behavior...
  Processing Dimensions protestors...
  Processing Cancelled Classes...
  Processing Green Key 2...
  Processing Sleep...
  Processing Boston Bombing...
  Processing Activity...
  Processing Comment...
  Processing Class 2...
  Processing Lab...
  Processing Mood 1...
  Processing PAM...
  Processing Social...
  Processing Administration response...
  Processing Exercise...
  Processing Class...
  Processing Green Key 1...
  Processing Mood 2...
  Processing Events...
  Processing Dimensions...
  Processing Mood...
  Processing QR_Code...
  Processing Dining Halls...
  Processing Study Spaces...
  Processing Dartmouth now...
  Processing Stress...

✓ Loaded 37,358 EMA responses
  (Skipped 12,154 responses with invalid/null question_ids)

EMA DATA LOADING COMPLETE!
